In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv("Mobile Reviews Sentiment null.csv")

In [3]:
print(df.shape)
print("\n",df.columns.tolist())
print("\n",df.dtypes)
print("\n",df.head(5))

(50000, 22)

 ['review_id', 'customer_name', 'age', 'brand', 'model', 'price_usd', 'price_local', 'currency', 'exchange_rate_to_usd', 'rating', 'sentiment', 'country', 'language', 'review_date', 'verified_purchase', 'battery_life_rating', 'camera_rating', 'performance_rating', 'design_rating', 'display_rating', 'helpful_votes', 'source']

 review_id                 int64
customer_name            object
age                       int64
brand                    object
model                    object
price_usd               float64
price_local              object
currency                 object
exchange_rate_to_usd    float64
rating                  float64
sentiment                object
country                  object
language                 object
review_date              object
verified_purchase          bool
battery_life_rating       int64
camera_rating             int64
performance_rating        int64
design_rating             int64
display_rating            int64
helpful_votes     

In [4]:
print(df.isnull().sum())

review_id                  0
customer_name              0
age                        0
brand                      0
model                      0
price_usd               2450
price_local             2431
currency                   0
exchange_rate_to_usd       0
rating                  2453
sentiment               2445
country                    0
language                   0
review_date                0
verified_purchase          0
battery_life_rating        0
camera_rating              0
performance_rating         0
design_rating              0
display_rating             0
helpful_votes              0
source                  2448
dtype: int64


In [5]:
print(df.duplicated().sum())

0


In [6]:
#missing value
#price_usd,rating
for col in ['price_usd','rating']:
    group_median=df.groupby(["brand","model"])[col].transform("median")
    df[col]=df[col].fillna(group_median)
    df[col]=df[col].fillna(df[col].median())#safety fallback

In [7]:
# price_local: convert to numeric, then re-derive from price_usd * exchange_rate where missing
df["price_local"] = pd.to_numeric(df["price_local"], errors="coerce")
missing_local = df["price_local"].isna()
df.loc[missing_local, "price_local"] = (
    df.loc[missing_local, "price_usd"] * df.loc[missing_local, "exchange_rate_to_usd"]
).round(2)

In [8]:
# sentiment: impute from the review's own numeric rating
def rating_to_sentiment(r):
    if r >= 4:
        return "Positive"
    elif r == 3:
        return "Neutral"
    return "Negative"

missing_sent = df["sentiment"].isna()
df.loc[missing_sent, "sentiment"] = df.loc[missing_sent, "rating"].apply(rating_to_sentiment)


In [9]:
# source: not predictive of product quality -> fill as "Unknown" rather than drop rows
df["source"] = df["source"].fillna("Unknown")

#print("Missing values after imputation:\n", df.isna().sum()[df.isna().sum() > 0])

In [10]:
print(df[['price_usd','rating','price_local','sentiment','source']].isnull().sum())

price_usd      0
rating         0
price_local    0
sentiment      0
source         0
dtype: int64


In [11]:
df.isnull().sum()

,0
review_id,0
customer_name,0
age,0
brand,0
model,0
price_usd,0
price_local,0
currency,0
exchange_rate_to_usd,0
rating,0


In [12]:
df["review_date"] = pd.to_datetime(df["review_date"],format="%m/%d/%Y",errors="coerce")

In [13]:
print(df[['review_date','price_local']].dtypes)

review_date    datetime64[ns]
price_local           float64
dtype: object


In [14]:
df["price_local"]=df["price_local"].astype(float)

In [15]:
# ---------------------------------------------------------------
# Select relevant features (price, ratings, specifications, engagement score)
# ---------------------------------------------------------------
df["engagement_score"] = df["helpful_votes"] + (df["verified_purchase"].astype(int) * 3)

numeric_features = ["price_usd", "rating", "battery_life_rating", "camera_rating",
                     "performance_rating", "design_rating", "display_rating",
                     "engagement_score"]
categorical_features = ["brand", "country", "model"]

model_df = df[numeric_features + categorical_features].copy()
print(model_df.shape)
print(model_df.head())

(50000, 11)
   price_usd  rating  battery_life_rating  camera_rating  performance_rating  \
0     337.31     2.0                    1              1                   3   
1     307.78     4.0                    3              2                   4   
2     864.53     4.0                    3              5                   3   
3     660.94     3.0                    1              3                   2   
4     792.13     3.0                    3              3                   2   

   design_rating  display_rating  engagement_score     brand country  \
0              2               1                 4    Realme   India   
1              3               2                 8    Realme  Brazil   
2              2               4                11    Google   India   
3              1               2                 3    Xiaomi     UAE   
4              2               1                 3  Motorola  Brazil   

           model  
0  Realme 12 Pro  
1  Realme 12 Pro  
2        Pixel 6 

In [16]:
df.columns.tolist()

['review_id',
 'customer_name',
 'age',
 'brand',
 'model',
 'price_usd',
 'price_local',
 'currency',
 'exchange_rate_to_usd',
 'rating',
 'sentiment',
 'country',
 'language',
 'review_date',
 'verified_purchase',
 'battery_life_rating',
 'camera_rating',
 'performance_rating',
 'design_rating',
 'display_rating',
 'helpful_votes',
 'source',
 'engagement_score']

In [17]:
df.head()

,review_id,customer_name,age,brand,model,price_usd,price_local,currency,exchange_rate_to_usd,rating,...,review_date,verified_purchase,battery_life_rating,camera_rating,performance_rating,design_rating,display_rating,helpful_votes,source,engagement_score
0,1,Aryan Maharaj,45,Realme,Realme 12 Pro,337.31,27996.73,INR,83.00,2.0,...,2023-11-06,True,1,1,3,2,1,1,Amazon,4
1,2,Davi Miguel Sousa,18,Realme,Realme 12 Pro,307.78,1754.35,BRL,5.70,4.0,...,2023-03-30,True,3,2,4,3,2,5,Flipkart,8
2,3,Pahal Balay,27,Google,Pixel 6,864.53,71755.99,INR,83.00,4.0,...,2022-12-07,True,3,5,3,2,4,8,AliExpress,11
3,4,David Guzman,19,Xiaomi,Redmi Note 13,660.94,2425.65,AED,3.67,3.0,...,2025-03-11,False,1,3,2,1,2,3,Amazon,3
4,5,Yago Leão,38,Motorola,Edge 50,792.13,4515.14,BRL,5.70,3.0,...,2023-09-29,True,3,3,2,2,1,0,BestBuy,3


In [18]:
# ---------------------------------------------------------------
# Encode categorical variables (brand, country, model)
# ---------------------------------------------------------------
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
encoded = encoder.fit_transform(model_df[categorical_features])
encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(categorical_features))

print(encoded_df.shape)
print(encoded_df.head())

(50000, 37)
   brand_Apple  brand_Google  brand_Motorola  brand_OnePlus  brand_Realme  \
0          0.0           0.0             0.0            0.0           1.0   
1          0.0           0.0             0.0            0.0           1.0   
2          0.0           1.0             0.0            0.0           0.0   
3          0.0           0.0             0.0            0.0           0.0   
4          0.0           0.0             1.0            0.0           0.0   

   brand_Samsung  brand_Xiaomi  country_Australia  country_Brazil  \
0            0.0           0.0                0.0             0.0   
1            0.0           0.0                0.0             1.0   
2            0.0           0.0                0.0             0.0   
3            0.0           1.0                0.0             0.0   
4            0.0           0.0                0.0             1.0   

   country_Canada  ...  model_Pixel 8  model_Poco X6  model_Razr 40  \
0             0.0  ...            0.0  

In [19]:
# ---------------------------------------------------------------
# Standardize and scale the dataset
# ---------------------------------------------------------------
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled = scaler.fit_transform(model_df[numeric_features])
scaled_df = pd.DataFrame(scaled, columns=numeric_features)

print(scaled_df.describe().round(2).loc[["mean", "std"]])

# Final model-ready matrix
final_df = pd.concat([scaled_df.reset_index(drop=True), encoded_df.reset_index(drop=True)], axis=1)
print(final_df.shape)

      price_usd  rating  battery_life_rating  camera_rating  \
mean        0.0    -0.0                  0.0            0.0   
std         1.0     1.0                  1.0            1.0   

      performance_rating  design_rating  display_rating  engagement_score  
mean                 0.0           -0.0            -0.0               0.0  
std                  1.0            1.0             1.0               1.0  
(50000, 45)


eda

In [20]:
import plotly.express as px


# Distribution across brands

brand_counts = df["brand"].value_counts().reset_index()
brand_counts.columns = ["brand", "count"]

fig1 = px.bar(brand_counts, x="brand", y="count", color="brand",
              title="Review Distribution by Brand")
fig1.show()


# Distribution across countries

country_counts = df["country"].value_counts().reset_index()
country_counts.columns = ["country", "count"]

fig2 = px.bar(country_counts, x="country", y="count", color="country",
              title="Review Distribution by Country")
fig2.show()

# Sunburst: same hierarchy, different visual style

brand_country = df.groupby(["brand", "country"]).size().reset_index(name="count")
fig3 = px.sunburst(brand_country, path=["brand", "country"], values="count",
                    title="Review Distribution: Brand > Country")
fig3.show()

In [31]:
import plotly.express as px

# ---------------------------------------------------------------
# Rank all products by average rating
# ---------------------------------------------------------------
product_ratings = df.groupby(["brand", "model"]).agg(
    avg_rating=("rating", "mean"),
    review_count=("review_id", "count")
).reset_index().sort_values("avg_rating", ascending=False)

product_ratings["label"] = product_ratings["brand"] + " " + product_ratings["model"]

# ---------------------------------------------------------------
# Full ranked bar chart, colored by brand
# ---------------------------------------------------------------
fig1 = px.bar(product_ratings, x="label", y="avg_rating", color="brand",
              orientation="v", title="All 22 Products Ranked by Average Rating")
fig1.update_layout(yaxis={"categoryorder": "total ascending"})
fig1.show()

# ---------------------------------------------------------------
# Top 5 vs Bottom 5, side by side
# ---------------------------------------------------------------
top5 = product_ratings.head(5)
bottom5 = product_ratings.tail(5)
compare = pd.concat([top5.assign(group="Top 5"), bottom5.assign(group="Bottom 5")])

fig2 = px.bar(compare, x="label", y="avg_rating", color="group",
              title="Top 5 vs Bottom 5 Rated Products")
fig2.update_layout(xaxis_tickangle=-30)
fig2.show()

In [32]:
import plotly.express as px
import plotly.graph_objects as go


# Full correlation heatmap

spec_cols = ["price_usd", "rating", "camera_rating", "performance_rating",
             "battery_life_rating", "display_rating", "design_rating"]

fig1 = px.imshow(df[spec_cols].corr().round(2), text_auto=True,
                  color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
                  title="Correlation: Price, Rating & Specifications")
fig1.show()


# Price bucket vs avg rating and avg specs (bar chart, clearer than scatter)

df["price_bucket"] = pd.cut(df["price_usd"], bins=5)
bucket_summary = df.groupby("price_bucket").agg(
    avg_rating=("rating", "mean"),
    avg_camera=("camera_rating", "mean"),
    avg_performance=("performance_rating", "mean"),
).reset_index()
bucket_summary["price_bucket"] = bucket_summary["price_bucket"].astype(str)

fig2 = px.bar(bucket_summary, x="price_bucket", y="avg_rating",
              title="Average Rating Across Price Tiers (Budget \u2192 Premium)")
fig2.show()


/tmp/ipykernel_719/974214184.py:26: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [23]:
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# CORRELATIONS: full numeric correlation matrix (incl. helpful_votes, age)

corr_cols = ["price_usd", "rating", "camera_rating", "performance_rating",
             "battery_life_rating", "display_rating", "design_rating",
             "helpful_votes", "age"]

fig1 = px.imshow(df[corr_cols].corr().round(2), text_auto=True,
                  color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
                  title="Full Correlation Heatmap")
fig1.show()

# TREND: review volume + average rating over time (dual axis)

df["review_date"] = pd.to_datetime(df["review_date"])
df["review_month"] = df["review_date"].dt.to_period("M").astype(str)
monthly = df.groupby("review_month").agg(
    review_count=("review_id", "count"),
    avg_rating=("rating", "mean")
).reset_index()

fig2 = make_subplots(specs=[[{"secondary_y": True}]])
fig2.add_trace(go.Bar(x=monthly["review_month"], y=monthly["review_count"],
                       name="Review Volume", opacity=0.4), secondary_y=False)
fig2.add_trace(go.Scatter(x=monthly["review_month"], y=monthly["avg_rating"],
                           name="Avg Rating", mode="lines+markers"), secondary_y=True)
fig2.update_layout(title="Review Volume & Average Rating Over Time")
fig2.show()


# PATTERN: the one real correlation - helpful_votes vs rating (r=0.45)

fig3 = px.scatter(df, x="rating", y="helpful_votes", trendline="ols", opacity=0.3,
                   title="Pattern: Helpful Votes Rise With Rating (r = 0.45)")
fig3.show()


# PATTERN: age group vs rating (mild, non-linear tendency)

df["age_group"] = pd.cut(df["age"], bins=[17, 25, 35, 45, 55, 100],
                          labels=["18-25", "26-35", "36-45", "46-55", "56+"])
age_pattern = df.groupby("age_group")["rating"].mean().reset_index()
fig4 = px.line(age_pattern, x="age_group", y="rating", markers=True,
               title="Rating Tendency by Age Group")
fig4.show()

/tmp/ipykernel_719/229897379.py:47: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [24]:

# 5. Statistical summaries and comparisons

print(df[["price_usd", "rating", "camera_rating", "performance_rating",
          "battery_life_rating", "display_rating", "design_rating"]].describe().round(2))

# Brand-level comparison table
brand_summary = df.groupby("brand").agg(
    avg_price=("price_usd", "mean"),
    avg_rating=("rating", "mean"),
    avg_camera=("camera_rating", "mean"),
    pct_positive=("sentiment", lambda s: (s == "Positive").mean() * 100),
    review_count=("review_id", "count"),
).round(2).sort_values("avg_rating", ascending=False)
print(brand_summary)

       price_usd    rating  camera_rating  performance_rating  \
count   50000.00  50000.00       50000.00            50000.00   
mean      689.80      3.12           2.72                2.72   
std       307.31      1.22           1.35                1.35   
min       180.02      1.00           1.00                1.00   
25%       450.80      2.00           1.00                1.00   
50%       640.80      3.00           3.00                3.00   
75%       901.40      4.00           4.00                4.00   
max      1499.89      5.00           5.00                5.00   

       battery_life_rating  display_rating  design_rating  
count             50000.00        50000.00       50000.00  
mean                  2.72            2.72           2.71  
std                   1.35            1.35           1.34  
min                   1.00            1.00           1.00  
25%                   1.00            1.00           1.00  
50%                   3.00            3.00           3

clustering

In [25]:
# ==================================================================
# 4. CLUSTERING (2D SEGMENTATION ANALYSIS: PRICE + REVIEW COUNT)
# ==================================================================
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import plotly.express as px

# ------------------------------------------------------------------
# Step 1: Aggregate reviews -> one row per product (brand + model)
# ------------------------------------------------------------------
product_df = df.groupby(["brand", "model"]).agg(
    avg_price_usd=("price_usd", "mean"),
    avg_rating=("rating", "mean"),
    review_count=("review_id", "count"),
).reset_index()

print("Aggregated product table:", product_df.shape)

# ------------------------------------------------------------------
# Step 2: Select 2D features & scale
# (Price + Review Count provides clear variance and avoids rating noise)
# ------------------------------------------------------------------
clustering_features = ["avg_price_usd", "review_count"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(product_df[clustering_features])

# ------------------------------------------------------------------
# Step 3: Apply K-Means with 4 clusters & Calculate Silhouette Score
# ------------------------------------------------------------------
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
product_df["cluster"] = kmeans.fit_predict(X_scaled)

# Calculate Silhouette Score on scaled 2D features
sil_score = silhouette_score(X_scaled, product_df["cluster"])
print(f"\n2D Silhouette Score: {sil_score:.3f}")

# ------------------------------------------------------------------
# Step 4: Analyze cluster profile to define segment labels
# ------------------------------------------------------------------
cluster_profile = product_df.groupby("cluster")[clustering_features].mean().round(2)
cluster_profile["product_count"] = product_df["cluster"].value_counts()
print("\nCluster profile (Mean Price & Review Count):")
print(cluster_profile.sort_values("avg_price_usd").to_string())

# Map cluster IDs to descriptive 2D segments based on price rank
price_rank = product_df.groupby("cluster")["avg_price_usd"].mean().sort_values()
tier_labels = ["Budget", "Mid-range", "Upper-mid", "Premium"]
segment_map = {cluster_id: tier_labels[i] for i, cluster_id in enumerate(price_rank.index)}

product_df["segment"] = product_df["cluster"].map(segment_map)

# ------------------------------------------------------------------
# Step 5: Display Product Segmentation Summary
# ------------------------------------------------------------------
print("\n--- 2D PRODUCT SEGMENTATION SUMMARY ---")
print(
    product_df[["brand", "model", "avg_price_usd", "review_count", "cluster", "segment"]]
    .sort_values("avg_price_usd", ascending=False)
    .to_string(index=False)
)

# ------------------------------------------------------------------
# Step 6: 2D Visualizations
# ------------------------------------------------------------------
# 2D Scatter Plot: Price vs. Review Count
fig1 = px.scatter(
    product_df,
    x="avg_price_usd",
    y="review_count",
    color="segment",
    hover_data=["brand", "avg_rating"],
    title=f"2D Product Clusters: Price vs. Review Count (K-Means k=4, Silhouette: {sil_score:.2f})"
)
fig1.update_traces(textposition="top center")
fig1.show()

# Price Distribution Boxplot
fig2 = px.box(
    product_df,
    x="segment",
    y="avg_price_usd",
    points="all",
    color="segment",
    title="Price Distribution by 2D Cluster"
)
fig2.show()

# Product Count per Cluster
fig3 = px.bar(
    product_df["segment"].value_counts().reset_index(),
    x="segment",
    y="count",
    color="segment",
    title="Products per 2D Cluster"
)
fig3.show()

Aggregated product table: (22, 5)

2D Silhouette Score: 0.718

Cluster profile (Mean Price & Review Count):
         avg_price_usd  review_count  product_count
cluster                                            
3               393.31       3566.00              2
0               478.22       2383.67              6
2               738.53       2395.00              6
1              1001.80       1774.50              8

--- 2D PRODUCT SEGMENTATION SUMMARY ---
   brand           model  avg_price_usd  review_count  cluster   segment
   Apple       iPhone 13    1106.224358          1767        1   Premium
   Apple       iPhone SE    1103.482906          1796        1   Premium
   Apple       iPhone 14    1101.632478          1788        1   Premium
   Apple   iPhone 15 Pro    1100.859370          1793        1   Premium
 Samsung      Galaxy A55     907.031930          1788        1   Premium
 Samsung  Galaxy Note 20     901.115207          1789        1   Premium
 Samsung   Galaxy Z Flip    

In [26]:
# Create a comprehensive summary table for business stakeholders
segment_profile = product_df.groupby("segment").agg(
    total_models=("model", "count"),
    min_price=("avg_price_usd", "min"),
    avg_price=("avg_price_usd", "mean"),
    max_price=("avg_price_usd", "max"),
    total_reviews=("review_count", "sum"),
    avg_review_count=("review_count", "mean"),
    avg_rating=("avg_rating", "mean")
).round(2).reset_index()

# Reorder logically from Budget to Premium
segment_order = ["Budget", "Mid-range", "Upper-mid", "Premium"]
segment_profile['segment'] = pd.Categorical(segment_profile['segment'], categories=segment_order, ordered=True)
segment_profile = segment_profile.sort_values('segment')

print(segment_profile.to_string(index=False))

  segment  total_models  min_price  avg_price  max_price  total_reviews  avg_review_count  avg_rating
   Budget             2     392.19     393.31     394.44           7132           3566.00        3.13
Mid-range             6     448.90     478.22     510.01          14302           2383.67        3.11
Upper-mid             6     670.12     738.53     808.63          14370           2395.00        3.12
  Premium             8     894.80    1001.80    1106.22          14196           1774.50        3.12


In [27]:
# 1. Export aggregated product segments
product_df.to_csv("product_segments_summary.csv", index=False)

# 2. Map segment labels back to the 5,000-row review-level dataset (if needed for review-level filtering)
df_final = df.merge(
    product_df[["brand", "model", "segment"]],
    on=["brand", "model"],
    how="left"
)
df_final.to_csv("reviews_with_product_segments.csv", index=False)

recommedation

In [29]:
# ==================================================================
# 5. SEGMENT-AWARE SIMILARITY-BASED RECOMMENDATION SYSTEM
# ==================================================================
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler

# ------------------------------------------------------------------
# Step 1: Feature Extraction & Merging Segment Labels
# ------------------------------------------------------------------
# 1. Aggregate specification ratings per product
spec_df = df.groupby(["brand", "model"]).agg(
    avg_price_usd=("price_usd", "mean"),
    avg_rating=("rating", "mean"),
    avg_battery_life=("battery_life_rating", "mean"),
    avg_camera=("camera_rating", "mean"),
    avg_performance=("performance_rating", "mean"),
    avg_design=("design_rating", "mean"),
    avg_display=("display_rating", "mean"),
    review_count=("review_id", "count")
).reset_index()

# 2. Merge segment labels from Section 4 (product_df)
rec_product_df = spec_df.merge(
    product_df[["brand", "model", "segment"]],
    on=["brand", "model"],
    how="left"
)

# ------------------------------------------------------------------
# Step 2: Feature Matrix Construction & Price Weighting
# ------------------------------------------------------------------
feature_cols = [
    "avg_price_usd", "avg_rating", "avg_battery_life",
    "avg_camera", "avg_performance", "avg_design", "avg_display"
]

# Standardize feature matrix
scaler = StandardScaler()
X_rec_scaled = scaler.fit_transform(rec_product_df[feature_cols])

# Apply double weight (2.5x) to Price so cost positioning dominates
# (Index 0 corresponds to 'avg_price_usd')
# X_rec_scaled[:, 0] = X_rec_scaled[:, 0] * 2.5

# Compute Weighted Cosine Similarity Matrix
similarity_matrix = cosine_similarity(X_rec_scaled)
similarity_df = pd.DataFrame(
    similarity_matrix,
    index=rec_product_df["model"],
    columns=rec_product_df["model"]
)

# ------------------------------------------------------------------
# Step 3: Segment-Aware Recommendation Engine
# ------------------------------------------------------------------
def recommend_similar_products(selected_model, top_n=3, strict_segment=True):
    """Recommends top_n products similar to selected_model based on
    weighted cosine similarity and optional segment boundary filtering.
    """
    # Strip leading/trailing whitespace from the selected_model
    selected_model = selected_model.strip()

    if selected_model not in similarity_df.index:
        available_models = list(similarity_df.index)
        return f"Error: '{selected_model}' not found. Available models: {available_models}"

    target_info = rec_product_df[rec_product_df["model"] == selected_model].iloc[0]
    target_segment = target_info["segment"]

    # Filter candidates by segment if strict_segment=True
    if strict_segment:
        candidate_mask = rec_product_df["segment"] == target_segment
        candidate_models = rec_product_df[candidate_mask]["model"]
    else:
        candidate_models = rec_product_df["model"]

    # Extract similarity scores for candidate products
    sim_scores = similarity_df[selected_model].loc[candidate_models].sort_values(ascending=False)

    # Exclude the target product itself and take top N
    recommended_models = sim_scores.drop(index=selected_model, errors='ignore').head(top_n)

    # Build structured validation table
    rec_results = []
    for model_name, score in recommended_models.items():
        row = rec_product_df[rec_product_df["model"] == model_name].iloc[0]
        rec_results.append({
            "Recommended Model": row["model"],
            "Brand": row["brand"],
            "Similarity Score": f"{score:.4f}",
            "Price (USD)": f"${row['avg_price_usd']:.2f}",
            "Price Delta": f"${row['avg_price_usd'] - target_info['avg_price_usd']:+.2f}",
            "Rating": f"{row['avg_rating']:.2f}"
        })

    print(f"=== RECOMMENDATIONS FOR: {selected_model} ===")
    print(f"Target Info -> Brand: {target_info['brand']} | Segment: {target_segment} | Price: ${target_info['avg_price_usd']:.2f} | Rating: {target_info['avg_rating']:.2f}\n")
    return pd.DataFrame(rec_results)

# ------------------------------------------------------------------
# Step 4: Validate Recommendations
# ------------------------------------------------------------------
# Example 1: Premium Flagship Test
print(recommend_similar_products(" iPhone 13 ", top_n=3, strict_segment=True).to_string(index=False))
print("\n" + "="*85 + "\n")

# Example 2: Budget Device Test
print(recommend_similar_products("Redmi Note 13", top_n=3, strict_segment=True).to_string(index=False))

=== RECOMMENDATIONS FOR: iPhone 13 ===
Target Info -> Brand: Apple | Segment: Premium | Price: $1106.22 | Rating: 3.15

Recommended Model   Brand Similarity Score Price (USD) Price Delta Rating
   Galaxy Note 20 Samsung           0.9462     $901.12    $-205.11   3.14
        iPhone 14   Apple           0.7980    $1101.63      $-4.59   3.18
    Galaxy Z Flip Samsung           0.4622     $899.22    $-207.01   3.13


=== RECOMMENDATIONS FOR: Redmi Note 13 ===
Target Info -> Brand: Xiaomi | Segment: Mid-range | Price: $449.68 | Rating: 3.11

Recommended Model    Brand Similarity Score Price (USD) Price Delta Rating
     Moto G Power Motorola           0.5754     $504.03     $+54.35   3.12
        Mi 13 Pro   Xiaomi           0.4040     $448.90      $-0.78   3.09
          Poco X6   Xiaomi           0.3567     $450.53      $+0.85   3.12
